# Extract CLIP Features cho MSRVTT

Notebook này extract CLIP ViT-B/32 visual features từ MSRVTT videos,
tạo ra file pickle cùng format với S3D features gốc của UniVL
nhưng với `dim=512` thay vì `dim=1024`.

## So sánh S3D vs CLIP

| | S3D | CLIP ViT-B/32 |
|--|-----|---------------|
| **Input** | `(B, 3, 16, 224, 224)` — clip 16 frames | `(B, 3, 224, 224)` — từng frame |
| **Output** | `(B, 1024)` — 1 vector/clip | `(B, 512)` — 1 vector/frame |
| **Video 30s @ 1fps** | shape `(2, 1024)` — 2 clips | shape `(30, 512)` — 30 frames |
| **Pretrain data** | Kinetics (action recognition) | 400M image-text pairs |
| **Semantic** | Motion-focused | Vision-Language aligned |

## Output format (giống S3D pickle gốc)
```python
{
    'video0':    np.array([num_frames, 512], dtype=float32),
    'video1':    np.array([num_frames, 512], dtype=float32),
    ...
    'video9999': np.array([num_frames, 512], dtype=float32),
}
```

## Chạy trên Google Colab (GPU T4)
- Runtime → Change runtime type → **GPU**
- Kết nối Google Drive chứa MSRVTT videos

---
## Bước 1: Kết nối Google Drive & Kiểm tra môi trường

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
import sys

print(f"Python: {sys.version}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
else:
    print("WARNING: No GPU — sẽ chậm hơn nhiều!")

## Bước 2: Cài đặt CLIP

In [ ]:
# CLIP của OpenAI — có pretrained weights, không cần train lại
!pip install git+https://github.com/openai/CLIP.git -q

import clip
print("CLIP version installed")
print("Available models:", clip.available_models())

## Bước 3: Cấu hình đường dẫn

Chỉnh `VIDEO_DIR` và `OUTPUT_PATH` cho phù hợp với cấu trúc Google Drive của bạn.

In [ ]:
import os
from pathlib import Path

# ===================== CẤU HÌNH =====================
# Đường dẫn thư mục chứa MSRVTT videos (.mp4)
# Theo cấu trúc VideoSwin notebook của bạn:
VIDEO_DIR = "/content/drive/MyDrive/PBL6/Data/Video_Datasets"  # Điều chỉnh nếu cần

# File output pickle — cùng format với msrvtt_videos_features.pickle gốc
OUTPUT_PATH = "/content/drive/MyDrive/PBL6/msrvtt_clip_vitb32_features.pickle"

# Model CLIP
CLIP_MODEL = "ViT-B/32"   # dim=512 | nếu GPU >12GB có thể dùng ViT-L/14 (dim=768)

# Số frames lấy mỗi giây (khớp với UniVL config: feature_framerate=1)
FEATURE_FRAMERATE = 1

# Batch size để encode frames qua CLIP (tùy VRAM)
# T4 15GB: batch_size=256 ổn
BATCH_SIZE = 256
# =====================================================

print(f"VIDEO_DIR   : {VIDEO_DIR}")
print(f"OUTPUT_PATH : {OUTPUT_PATH}")
print(f"CLIP model  : {CLIP_MODEL}")
print(f"Feature fps : {FEATURE_FRAMERATE}")
print(f"Batch size  : {BATCH_SIZE}")

# Kiểm tra thư mục videos
video_files = list(Path(VIDEO_DIR).rglob("*.mp4"))
print(f"\nTìm thấy {len(video_files)} video .mp4")
if video_files:
    print(f"Ví dụ: {video_files[0]}")

## Bước 4: Load CLIP Model

CLIP sẽ **tự động download pretrained weights** (~350MB cho ViT-B/32) lần đầu chạy.

In [ ]:
import clip
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Load CLIP — tự download pretrained weights lần đầu
print(f"Loading CLIP {CLIP_MODEL} on {DEVICE}...")
clip_model, preprocess = clip.load(CLIP_MODEL, device=DEVICE)
clip_model.eval()

# Lấy output dimension
FEATURE_DIM = clip_model.visual.output_dim
print(f"Done!")
print(f"  Visual input : (B, 3, 224, 224)  ← preprocess() tạo ra")
print(f"  Visual output: (B, {FEATURE_DIM})  ← encode_image() trả về")
print(f"  Feature dim  : {FEATURE_DIM}")
print(f"  Param count  : {sum(p.numel() for p in clip_model.visual.parameters()) / 1e6:.1f}M (visual encoder)")

## Bước 5: Hàm Extract Frames từ Video

Tương tự `extract_frames_from_video` trong VideoSwin notebook,
nhưng output là danh sách PIL Images để `preprocess()` của CLIP xử lý.

In [ ]:
import cv2
import numpy as np
from PIL import Image

def extract_frames_for_clip(video_path, framerate=1):
    """
    Extract frames từ video theo framerate nhất định.
    Trả về list PIL Images để CLIP preprocess.

    So sánh với VideoSwin notebook:
      VideoSwin: trả về np.array (num_frames, 224, 224, 3) — TF channels-last
      CLIP:      trả về list PIL Image                     — CLIP tự resize/normalize

    Args:
        video_path (str): Đường dẫn video .mp4
        framerate (int): Số frames lấy mỗi giây (default=1, khớp UniVL)

    Returns:
        frames (list[PIL.Image]): Danh sách frames
        video_id (str): ID video (tên file không có .mp4)
    """
    cap = cv2.VideoCapture(str(video_path))

    if not cap.isOpened():
        print(f"  Cannot open: {video_path}")
        return [], ""

    video_fps = cap.get(cv2.CAP_PROP_FPS)
    if video_fps <= 0:
        video_fps = 30.0  # fallback

    # Lấy 1 frame mỗi (video_fps / framerate) frames
    sample_every = max(1, int(round(video_fps / framerate)))

    frames = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if frame_idx % sample_every == 0:
            # BGR → RGB → PIL Image
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(Image.fromarray(frame_rgb))
        frame_idx += 1

    cap.release()
    video_id = Path(video_path).stem  # "video0" từ "video0.mp4"
    return frames, video_id


# --- Test với 1 video ---
if video_files:
    test_frames, test_id = extract_frames_for_clip(video_files[0], framerate=FEATURE_FRAMERATE)
    print(f"Test video   : {video_files[0].name}")
    print(f"Video ID     : {test_id}")
    print(f"Frames count : {len(test_frames)}")
    if test_frames:
        print(f"Frame size   : {test_frames[0].size}  (PIL Image, W x H)")
        # Sau preprocess của CLIP:
        t = preprocess(test_frames[0])
        print(f"After CLIP preprocess: {tuple(t.shape)}  → (3, 224, 224)")

## Bước 6: Hàm Extract CLIP Features cho 1 video

```
Video .mp4
   ↓  extract_frames_for_clip()
List[PIL Image]  (num_frames ảnh)
   ↓  preprocess()  — CLIP's own transform: resize 224, center-crop, normalize
Tensor (num_frames, 3, 224, 224)
   ↓  clip_model.encode_image()  — ViT-B/32 forward pass
Tensor (num_frames, 512)  float16
   ↓  .float().cpu().numpy()
np.array (num_frames, 512)  float32   ← LƯU VÀO PICKLE
```

In [ ]:
import torch
import numpy as np

def extract_clip_features_one_video(frames, clip_model, preprocess, device, batch_size=256):
    """
    Extract CLIP features từ danh sách PIL frames.

    Input shape (conceptual):
        frames: List[PIL.Image]  — num_frames ảnh

    CLIP internal:
        preprocess(frame) → Tensor(3, 224, 224)
        encode_image(batch) với batch shape (B, 3, 224, 224) → (B, 512) float16

    Output shape:
        np.array (num_frames, 512) float32
        → GIỐNG FORMAT S3D: (num_clips, 1024), chỉ khác dim

    Args:
        frames: list PIL Images từ extract_frames_for_clip()
        batch_size: số frames xử lý 1 lần qua GPU

    Returns:
        features: np.array (num_frames, 512) float32
    """
    if not frames:
        return np.zeros((1, clip_model.visual.output_dim), dtype=np.float32)

    # Preprocess tất cả frames: List[PIL] → Tensor(N, 3, 224, 224)
    frame_tensors = torch.stack([preprocess(f) for f in frames])  # (N, 3, 224, 224)

    feat_dim = clip_model.visual.output_dim
    all_features = []

    with torch.no_grad():
        for i in range(0, len(frame_tensors), batch_size):
            batch = frame_tensors[i : i + batch_size].to(device)  # (B, 3, 224, 224)
            feats = clip_model.encode_image(batch)                 # (B, 512) float16
            all_features.append(feats.float().cpu())              # convert float32

    features = torch.cat(all_features, dim=0).numpy()  # (num_frames, 512)
    return features.astype(np.float32)


# --- Test với video ở Bước 5 ---
if video_files and test_frames:
    feat = extract_clip_features_one_video(test_frames, clip_model, preprocess, DEVICE, BATCH_SIZE)
    print(f"CLIP feature shape: {feat.shape}")
    print(f"  → (num_frames={feat.shape[0]}, dim={feat.shape[1]})")
    print(f"  Dtype: {feat.dtype}")
    print(f"  Value range: [{feat.min():.3f}, {feat.max():.3f}]")
    print(f"\nSo sánh với S3D gốc:")
    print(f"  S3D  : (num_clips, 1024)  — num_clips = ceil(num_frames / 16) = ceil({len(test_frames)}/16) = {int(np.ceil(len(test_frames)/16))}")
    print(f"  CLIP : (num_frames, 512)  — num_frames = {feat.shape[0]}")

## Bước 7: Extract CLIP Features cho toàn bộ MSRVTT Dataset

Ước tính thời gian:
- MSRVTT có ~10,000 videos
- Mỗi video ~15s @ 1fps = 15 frames
- T4 GPU: ~2-3 video/giây → **~1-2 giờ**

In [ ]:
import pickle
import time
from tqdm import tqdm
from pathlib import Path

def extract_clip_features_dataset(video_dir, output_path, clip_model, preprocess,
                                   device, framerate=1, batch_size=256):
    """
    Extract CLIP features cho toàn bộ dataset và lưu vào pickle.

    Output pickle format — GIỐNG HỆT msrvtt_videos_features.pickle gốc:
        {
            'video0':    np.array (num_frames, 512) float32,
            'video1':    np.array (num_frames, 512) float32,
            ...
        }
    Chỉ khác: dim=512 thay vì dim=1024 của S3D

    Args:
        video_dir: Thư mục chứa video .mp4 (tìm đệ quy)
        output_path: Đường dẫn file .pickle đầu ra
    """
    all_video_files = sorted(Path(video_dir).rglob("*.mp4"))
    print(f"Tìm thấy {len(all_video_files)} videos trong {video_dir}")

    if not all_video_files:
        print("ERROR: Không tìm thấy video nào!")
        return {}

    all_features = {}
    failed = []
    start_time = time.time()

    for video_path in tqdm(all_video_files, desc="Extracting CLIP features"):
        # Extract frames
        frames, video_id = extract_frames_for_clip(video_path, framerate=framerate)

        if not frames:
            failed.append(str(video_path))
            feat_dim = clip_model.visual.output_dim
            all_features[video_id] = np.zeros((1, feat_dim), dtype=np.float32)
            continue

        # Extract CLIP features
        features = extract_clip_features_one_video(
            frames, clip_model, preprocess, device, batch_size
        )
        all_features[video_id] = features

    elapsed = time.time() - start_time
    print(f"\nHoàn thành trong {elapsed/60:.1f} phút")
    print(f"  Thành công : {len(all_features) - len(failed)}/{len(all_video_files)} videos")
    print(f"  Thất bại   : {len(failed)} videos")
    if failed:
        print(f"  Videos lỗi : {failed[:5]} ...")

    # Lưu pickle
    print(f"\nLưu vào {output_path} ...")
    with open(output_path, 'wb') as f:
        pickle.dump(all_features, f, protocol=4)

    size_mb = Path(output_path).stat().st_size / 1024**2
    print(f"Done! File size: {size_mb:.1f} MB")
    return all_features


print("Hàm extract_clip_features_dataset() đã sẵn sàng")

## Bước 8: Chạy Extract (toàn bộ dataset)

> **Lưu ý**: Cell này chạy ~1-2 giờ trên T4. Colab có thể timeout sau 12h.

In [ ]:
all_features = extract_clip_features_dataset(
    video_dir=VIDEO_DIR,
    output_path=OUTPUT_PATH,
    clip_model=clip_model,
    preprocess=preprocess,
    device=DEVICE,
    framerate=FEATURE_FRAMERATE,
    batch_size=BATCH_SIZE
)

## Bước 9: Kiểm tra kết quả

Xác nhận output pickle có format đúng để dùng với UniVL.

In [ ]:
import pickle
import numpy as np
from pathlib import Path

# Load lại để kiểm tra
with open(OUTPUT_PATH, 'rb') as f:
    features = pickle.load(f)

print(f"=== Kiểm tra CLIP features pickle ===")
print(f"Tổng số videos : {len(features)}")

# Thống kê shape
shapes = [v.shape for v in features.values()]
num_frames_list = [s[0] for s in shapes]
dims = set(s[1] for s in shapes)

print(f"Feature dim    : {dims}  (phải là {{512}} cho ViT-B/32)")
print(f"Num frames     : min={min(num_frames_list)}, max={max(num_frames_list)}, avg={np.mean(num_frames_list):.1f}")
print(f"Dtype          : {list(features.values())[0].dtype}  (phải là float32)")

# Xem 3 video mẫu
print(f"\nMẫu 3 videos:")
for vid_id, feat in list(features.items())[:3]:
    print(f"  {vid_id}: shape={feat.shape}, range=[{feat.min():.3f}, {feat.max():.3f}]")

# So sánh với S3D gốc
print(f"\n=== So sánh với S3D features gốc ===")
print(f"S3D  : {{video_id: np.array([num_clips, 1024])}}  — dim=1024")
print(f"CLIP : {{video_id: np.array([num_frames, 512])}}  — dim=512")
print(f"\nUniVL training command cần đổi: --video_dim 1024 → --video_dim 512")

## Bước 10: Lưu ý khi dùng với UniVL

Sau khi có file `msrvtt_clip_vitb32_features.pickle`, chỉ cần thay đổi 2 tham số trong training command:

```bash
# TRƯỚC (S3D features):
--features_path data/msrvtt/msrvtt_videos_features.pickle \
--video_dim 1024

# SAU (CLIP features):
--features_path data/msrvtt/msrvtt_clip_vitb32_features.pickle \
--video_dim 512
```

Không cần thay đổi gì trong `modules/modeling.py` hay dataloader —
`video_dim` được truyền qua args và config tự động adapt.